# `ptof_obs_setup_seed`

## What this notebook does
Hand-maintained reference-data seeding and one-off maintenance for the observability pipeline.
It is **not part of the scheduled `obs_fresh_scan` job** -- it's run manually, by a human, when a
new capability needs registering, a threshold needs recording, or a one-time cleanup/backfill is
needed. Every other notebook in this repo *reads* the tables this notebook seeds; nothing else
writes to them.

## Position in the pipeline
- **Not in any job DAG.** Run interactively/manually, occasionally, not on a schedule.
- **Downstream readers:** `ptof_obs_latency_detection`, `ptof_obs_mal_output`,
  `ptof_obs_behavioral_correlation`, and `ptof_obs_hallucination_detection` all join against
  `capability_registry` and/or `runtime_allowlist` as ground-truth reference data.
  `ptof_obs_alert.ipynb` reads `threshold_basis` for the "why this number" documentation surfaced
  on Teams cards, and creates/updates `obs_incidents` (this notebook only creates the table once).
  `ptof_obs_weekly_runtime_digest` reads/writes `runtime_observed`.

## Write safety (Phase 5 of `agent_obs_implementation_plan.md` -- done)
`runtime_allowlist` and `capability_registry` are the only tables in this system that encode human
judgment -- everything else is derived/recomputed from raw call data. Every write cell below is now
either schema-only (`CREATE TABLE IF NOT EXISTS`) or a guarded insert (`WHERE NOT EXISTS` /
`LEFT ANTI JOIN`, keyed on the table's natural key) -- re-running this notebook is a no-op against
rows that already exist and cannot wipe a hand-curated disposition. The two remaining non-idempotent
cells are intentionally one-time and are marked as such.

| Cell | Writes | Idempotent? |
|---|---|---|
| 1 | `runtime_allowlist` | Yes -- `CREATE TABLE IF NOT EXISTS` + anti-join insert on (environment, capability) |
| 2 | `capability_registry` schema | Yes -- `CREATE TABLE IF NOT EXISTS` |
| 3 | `capability_registry` | Yes -- anti-join insert on capability |
| 4 | `transport_allowlist`, `success_rate_daily` (drop) | Yes -- both `DROP TABLE IF EXISTS`, but this is one-time cleanup kept for the record, not something to run again on purpose |
| 5 | (read-only) | n/a |
| 6 | `obs_incidents` schema | Yes -- `CREATE TABLE IF NOT EXISTS` |
| 7 | `obs_incidents` (UPDATE) | **No -- one-time, date-bounded.** Re-running is safe only because the bound (`first_detected < 2026-08-21`) is fixed; do not remove the date bound |
| 8 | `_obs_watermark` | Yes -- `CREATE TABLE IF NOT EXISTS` + `WHERE NOT EXISTS` seed |
| 9 | `threshold_basis` | Yes -- `CREATE TABLE IF NOT EXISTS` + anti-join insert on check_name |
| 10 | `runtime_observed` schema | Yes -- `CREATE TABLE IF NOT EXISTS` |
| 11 | `threshold_basis` | Yes -- `WHERE NOT EXISTS` per check_name |
| 12 | `threshold_basis` | Yes -- `WHERE NOT EXISTS` per check_name |
| 13 | `capability_registry` | Yes -- `WHERE NOT EXISTS` per capability |
| 14 | `runtime_allowlist` | Yes -- `WHERE NOT EXISTS` per (environment, capability) |

## Tables/views touched
- **Writes:** `runtime_allowlist` (human-curated: which transport/model_config/scheduler_run
  combinations are sanctioned per capability/environment), `capability_registry` (human-curated:
  per-capability metadata -- is it generative, is it GxP-relevant, who owns it, is it active),
  `obs_incidents` (the append-only finding record every detector's alert path writes to --
  created here, populated by `ptof_obs_alert.ipynb`), `_obs_watermark` (incremental-processing
  bookmark for `faithfulness_scores` in the hallucination pipeline), `threshold_basis`
  (documents what every detection threshold in this system is, why it's set where it is, and
  whether it's backed by real evidence yet), `runtime_observed` (automatically-populated log of
  transport/model_config combinations actually seen, kept separate from what's *permitted* in
  `runtime_allowlist`).
- **Reads:** the tables above, for idempotency checks and summary counts.


In [ ]:
%sql
-- SAFE to re-run: CREATE TABLE IF NOT EXISTS + anti-join insert, guarded per (environment,
-- capability). Phase 5 of agent_obs_implementation_plan.md: this used to be CREATE OR REPLACE
-- TABLE, which meant re-running this notebook could silently wipe a row a human had hand-edited
-- directly in the table. The anti-join makes re-running this cell a no-op once these rows exist,
-- without changing what the table contains today.
--
-- runtime_allowlist -- keyed on capability, not scheduler_run.
-- 'live' contains both June's summary on demo-claude-sonnet-4-6-pwc-omi and August's
-- dsa_optimize on test-bot-claude-v1, so scheduler_run does not discriminate.
CREATE TABLE IF NOT EXISTS mq_gmdf_dev.oil_obs.runtime_allowlist (
    environment            STRING,
    capability             STRING,
    allowed_transports     ARRAY<STRING>,
    allowed_model_configs  ARRAY<STRING>,
    allowed_scheduler_runs ARRAY<STRING>
);

INSERT INTO mq_gmdf_dev.oil_obs.runtime_allowlist
SELECT t.* FROM VALUES
  ('dev', 'dsa_copilot',        array('cortex'), array('vertex26'),                      array('live')),
  ('dev', 'dsa_compare',        array('cortex'), array('vertex26'),                      array('live')),
  ('dev', 'dsa_batch_summary',  array('cortex'), array('vertex26'),                      array('live')),
  ('dev', 'dsa_optimize',       array('cortex'), array('vertex26','test-bot-claude-v1'), array('live')),
  ('dev', 'summary',            array('cortex'),
          array('mq-ai-dg-poc','ptof-ish-handover-v1-das','demo-claude-sonnet-4-6-pwc-omi'),
          array('live','final_handover','pre_handover')),
  ('dev', 'watchout_narratives',array('cortex'), array('demo-claude-sonnet-4-6-pwc-omi'), array('live')),
  ('prod','dsa_copilot',        array('cortex'), array('vertex26'), array('live')),
  ('prod','dsa_compare',        array('cortex'), array('vertex26'), array('live')),
  ('prod','dsa_batch_summary',  array('cortex'), array('vertex26'), array('live')),
  ('prod','dsa_optimize',       array('cortex'), array('vertex26'), array('live')),
  ('prod','summary',            array('cortex'), array('mq-ai-dg-poc','ptof-ish-handover-v1-das'),
                                                 array('final_handover')),
  ('prod','watchout_narratives',array('cortex'), array('vertex26'), array('live'))
AS t(environment, capability, allowed_transports, allowed_model_configs, allowed_scheduler_runs)
LEFT ANTI JOIN mq_gmdf_dev.oil_obs.runtime_allowlist existing
  ON existing.environment = t.environment AND existing.capability = t.capability;

In [0]:
%sql
-- SAFE to re-run: CREATE TABLE IF NOT EXISTS is a no-op once the table exists. This just
-- declares the schema for capability_registry -- the actual human-curated rows are seeded in the
-- next cell.
CREATE TABLE IF NOT EXISTS mq_gmdf_dev.oil_obs.capability_registry (
    capability           STRING,
    is_generative        BOOLEAN,
    is_gxp_relevant      BOOLEAN,
    expected_min_daily   INT,
    required_fields      ARRAY<STRING>,
    owner                STRING,
    active               BOOLEAN,
    notes                STRING,
    silence_grace_hours  INT
);

In [ ]:
%sql
-- SAFE to re-run: anti-join insert guarded per capability. Phase 5 of
-- agent_obs_implementation_plan.md: this used to be INSERT OVERWRITE, which replaced every row
-- on every run -- re-running this notebook could silently wipe hand-curated dispositions (owner,
-- active, is_gxp_relevant) that a human had since edited directly in the table. The anti-join
-- makes re-running this cell a no-op once these rows exist, without changing what the table
-- contains today. (The CREATE TABLE IF NOT EXISTS for this table is in the cell above.)
--
-- is_gxp_relevant = true only for SUMMARIZATION capabilities, where every number in the output
-- should trace to an input. Chat, comparative scoring, and projection legitimately emit numbers
-- absent from their prompts, so token-grounding does not apply to them.
INSERT INTO mq_gmdf_dev.oil_obs.capability_registry
SELECT t.* FROM VALUES
  ('dsa_copilot',        true,  false, 20, array(),             'dsa-team', true,
     'vertex26 · conversational chat surface — user_prompt is a turn, not a grounding block', 26),
  ('dsa_optimize',       true,  false, 10, array(),             'dsa-team', true,
     'test-bot-claude-v1 · 100+/100+ failing · emits computed projection deltas, grounding N/A', 2),
  ('dsa_compare',        true,  false,  0, array(),             'dsa-team', true,
     'vertex26 · comparative scoring — emits its own rank/score values, grounding N/A', NULL),
  ('dsa_batch_summary',  true,  true,   0, array(),             'dsa-team', true,
     'vertex26 · summarization — grounding diff applies', NULL),
  ('watchout_narratives',true,  true,   0, array(),             'ish-team', true,
     'single call 2026-08-12 · summarization', NULL),
  ('summary',            false, true,   0, array('how_we_ran'), 'ish-team', false,
     'silent since 2026-06-12 · 22 auth failures during Jun 11 migration', NULL)
AS t(capability, is_generative, is_gxp_relevant, expected_min_daily, required_fields, owner,
     active, notes, silence_grace_hours)
LEFT ANTI JOIN mq_gmdf_dev.oil_obs.capability_registry existing
  ON existing.capability = t.capability;

In [0]:
%sql
-- ONE-TIME cleanup, safe to re-run only because both DROPs are IF EXISTS (a second run is a
-- harmless no-op, not because dropping tables is generally idempotent-safe).
-- Executed 2026-08-19. Kept for the record; safe to re-run.
DROP TABLE IF EXISTS mq_gmdf_dev.oil_obs.transport_allowlist;  -- stale, scheduler_run-keyed, re-seeded by task 03
DROP TABLE IF EXISTS mq_gmdf_dev.oil_obs.success_rate_daily;   -- orphaned ad-hoc, single write 2026-08-18

In [0]:
%sql
-- SAFE to re-run: read-only sanity check. Confirms the seed rows above landed with the expected
-- per-environment / per-gxp-flag row counts before anything downstream depends on them.
SELECT 'runtime_allowlist' AS t, environment AS k, count(*) AS n
FROM mq_gmdf_dev.oil_obs.runtime_allowlist GROUP BY 1,2
UNION ALL
SELECT 'capability_registry', concat('gxp=', cast(is_gxp_relevant AS STRING)), count(*)
FROM mq_gmdf_dev.oil_obs.capability_registry GROUP BY 1,2
ORDER BY t, k;

t,k,n
capability_registry,gxp=false,3
capability_registry,gxp=true,3
runtime_allowlist,dev,6
runtime_allowlist,prod,6


In [0]:
%sql
-- SAFE to re-run: CREATE TABLE IF NOT EXISTS. This is the single table every detector's finding
-- ultimately lands in, and the only table ptof_obs_alert.ipynb's Teams-notify query reads from.
--
-- obs_incidents — append-only finding record.
-- MERGE keyed on (detector, source_row_id) so re-detecting the same finding updates rather than
-- duplicates. That gives dedup for free and makes acknowledgement stick across runs.
CREATE TABLE IF NOT EXISTS mq_gmdf_dev.oil_obs.obs_incidents (
    detector         STRING,        -- which detector produced this
    source_row_id    STRING,        -- the id being flagged; stable across runs
    capability       STRING,
    severity         STRING,        -- CRITICAL | WARN | INFO
    first_detected   TIMESTAMP,     -- set once, never updated
    last_detected    TIMESTAMP,     -- refreshed each time the finding is still present
    detection_count  BIGINT,        -- how many runs have seen it
    signal_payload   STRING,        -- JSON detail for triage
    notified_at      TIMESTAMP,     -- stamped when the alert reports it
    acknowledged_by  STRING,        -- set by a human, by hand
    acknowledged_at  TIMESTAMP,
    resolved_at      TIMESTAMP      -- set by hand once remediated
) CLUSTER BY (first_detected, detector);

In [0]:
%sql
-- ONE-TIME, date-bounded data mutation -- NOT safe to blindly re-run indefinitely. Acknowledging
-- an incident here means ptof_obs_alert.ipynb's Teams-notify query will never re-fire it, so a
-- careless unbounded version of this UPDATE would permanently silence a real class of failure.
--
-- One-off: acknowledge the 15 SMTP failures known and escalated as of 2026-08-20.
-- Bounded by date on purpose. Unbounded, this acknowledges FUTURE failures on any re-run,
-- suppressing their notification and falsifying the disposition record.
UPDATE mq_gmdf_dev.oil_obs.obs_incidents
SET acknowledged_by = 'tyler.kei@lilly.com',
    acknowledged_at = current_timestamp()
WHERE detector = 'handover_delivery'
  AND acknowledged_at IS NULL
  AND first_detected < TIMESTAMP '2026-08-21 00:00:00';

num_affected_rows
0


In [0]:
%sql
-- SAFE to re-run: table creation is IF NOT EXISTS, and the seed insert below is itself
-- WHERE-NOT-EXISTS-guarded. This is the incremental-processing bookmark that lets
-- ptof_obs_hallucination_detection's faithfulness_scores compute only new rows each run instead
-- of rescanning all of v_llm_bronze -- the one detector in this codebase using the
-- watermark+MERGE incremental pattern (see Future Phase C of the implementation plan, which wants
-- to generalize this pattern to every other detector).
CREATE TABLE IF NOT EXISTS mq_gmdf_dev.oil_obs._obs_watermark (
    detector          STRING,
    last_processed_ts TIMESTAMP,
    updated_at        TIMESTAMP
);

-- Seed only if absent. INSERT OVERWRITE rewinds the watermark 24h on every seed run, silently
-- re-scoring a day of rows and masking whether task 04's advance cell is working.
INSERT INTO mq_gmdf_dev.oil_obs._obs_watermark
SELECT 'faithfulness_scores', current_timestamp() - INTERVAL 24 HOURS, current_timestamp()
WHERE NOT EXISTS (SELECT 1 FROM mq_gmdf_dev.oil_obs._obs_watermark
                  WHERE detector = 'faithfulness_scores');

num_affected_rows,num_inserted_rows
0,0


In [ ]:
%sql
-- SAFE to re-run: CREATE TABLE IF NOT EXISTS + anti-join insert, guarded per check_name. Phase 5
-- of agent_obs_implementation_plan.md: this used to be CREATE OR REPLACE TABLE, which discarded
-- every row -- including the ones cells 11/12 below insert with their own WHERE NOT EXISTS
-- guards -- on every re-run. The anti-join makes re-running this cell a no-op once these rows
-- exist, without changing what the table contains today.
--
-- threshold_basis — what each threshold is, why, and whether evidence supports it.
-- Every number in this pipeline is provisional: there are no labelled examples yet, so these are
-- anchored to observed ranges rather than to confirmed incidents.
CREATE TABLE IF NOT EXISTS mq_gmdf_dev.oil_obs.threshold_basis (
    check_name STRING,
    threshold  STRING,
    basis      STRING,
    set_on     STRING,
    status     STRING
);

INSERT INTO mq_gmdf_dev.oil_obs.threshold_basis
SELECT t.* FROM VALUES
  ('write_lag', 'p95_ingest_only_s > 10',
   'observed 2.1-2.3s across all hours; ~4x headroom', '2026-08-20', 'provisional'),
  ('capability_error_rate', 'error_rate = 1.0 & n>=5, or > 0.20 & n>=10',
   'observed 0.00 or 0.93-1.00; no middle ground in data', '2026-08-20', 'provisional'),
  ('capability_silence dsa_copilot', 'silence_grace_hours = 26',
   'raised from 18 after two false breaches (19.5h, 22.7h), zero true', '2026-08-20', 'revised'),
  ('capability_silence dsa_optimize', 'silence_grace_hours = 2',
   'automated hourly poll; 2h silence unambiguous', '2026-08-20', 'provisional'),
  ('hallucination medium', 'pctile < 0.05 AND similarity < 0.75',
   'similarity band 0.692-0.887 over 136 rows; floor excludes single-row capabilities',
   '2026-08-20', 'unvalidated - no row qualifies'),
  ('ungrounded_token_count', '> 3 AND is_gxp_relevant',
   'only 3 low-volume capabilities are gxp_relevant; untested where it applies',
   '2026-08-20', 'unvalidated'),
  ('handover_delivery_rate', 'failure_pct_7d > 20 AND attempts >= 10',
   'all-time baseline 9.3% (15 of 162 since Jun 19); currently 12.5% over 7d',
   '2026-08-20', 'provisional'),
  ('latency_anomaly', 'p95 + 3*IQR, is_reliable only',
   'no capability qualifies yet (needs n>=30 AND span>=7d); live ~2026-08-25',
   '2026-08-20', 'not yet active'),
  ('long_running_incident', 'detection_count >= 20 AND age >= 24h',
   'detection_count counts RUNS that saw it, not occurrences — at 5-min triggering this trips '
   'in under 2h. Needs raising to ~200 or switching to age-based',
   '2026-08-20', 'known miscalibrated')
AS t(check_name, threshold, basis, set_on, status)
LEFT ANTI JOIN mq_gmdf_dev.oil_obs.threshold_basis existing
  ON existing.check_name = t.check_name;

In [0]:
%sql
-- SAFE to re-run: CREATE TABLE IF NOT EXISTS, schema only. This is the automatically-populated
-- log of transport/model_config combinations actually observed in traffic, deliberately kept
-- separate from runtime_allowlist (which stays 100% human-curated) so "what we saw" and "what we
-- decided to permit" never get conflated. ptof_obs_weekly_runtime_digest.ipynb is what writes
-- rows here and reads digest_reported_at/disposition back out.
--
-- runtime_observed — a record of what has been SEEN, kept separate from what is PERMITTED.
-- runtime_allowlist stays human-seeded. This table is written automatically; disposition is not.
CREATE TABLE IF NOT EXISTS mq_gmdf_dev.oil_obs.runtime_observed (
    environment          STRING,
    violation_signature  STRING,
    capability           STRING,
    transport            STRING,
    model_config         STRING,
    scheduler_run        STRING,
    violation_type       STRING,
    violation_tier       STRING,
    first_seen           TIMESTAMP,
    last_seen            TIMESTAMP,
    occurrences_7d       BIGINT,
    digest_reported_at   TIMESTAMP,
    disposition          STRING,   -- NULL = undispositioned | sanctioned | rejected | transient
    dispositioned_by     STRING,
    dispositioned_at     TIMESTAMP
);

In [0]:
%sql
-- SAFE to re-run: WHERE NOT EXISTS-guarded on check_name, the idempotent pattern this table
-- should have used from the start (see the "ONE-TIME, destructive" warning on the CREATE OR
-- REPLACE cell above -- this is the fix, applied going forward rather than retroactively).
--
-- Threshold provenance. Conditional insert, because INSERT INTO in anything re-runnable is
-- what put 260 rows across 52 runs into a reference table (§15 lesson 5).
INSERT INTO mq_gmdf_dev.oil_obs.threshold_basis
  (check_name, threshold, basis, set_on, status)
SELECT * FROM VALUES
  ('runtime_violation_immediate',
   'transport OR model_config sanctioned for no capability in env',
   'A new capability on an already-sanctioned transport + model_config is a paperwork lag. A new transport or model config is a different claim about the system. Tier split, not a numeric threshold. scheduler_run excluded: live spans two agent generations and does not discriminate.',
   DATE '2026-08-27', 'provisional'),
  ('runtime_violation_digest_cadence',
   'weekly, Monday 08:00 America/Indianapolis',
   'Unlisted capabilities are normal in ISH: the allowlist is maintained more slowly than the agent ships. Replaces 529 CRITICAL cards for one configuration.',
   DATE '2026-08-27', 'provisional'),
  ('runtime_observed_digest_floor',
   'occurrences_7d >= 5',
   'Suppresses a single stray call from the digest body. Explicitly NOT an auto-approval threshold: occurrence count does not discriminate sanctioned from misconfigured. dsa_optimize at 0/121 would clear any floor within a day.',
   DATE '2026-08-27', 'unvalidated')
  AS t(check_name, threshold, basis, set_on, status)
WHERE NOT EXISTS (
  SELECT 1 FROM mq_gmdf_dev.oil_obs.threshold_basis
  WHERE check_name = 'runtime_violation_immediate');

num_affected_rows,num_inserted_rows
3,3


In [ ]:
%sql
-- SAFE to re-run: WHERE NOT EXISTS-guarded per check_name, same idempotent pattern as the
-- cell above. Phase 4 items 1-2 of agent_obs_implementation_plan.md: two known, already-
-- accepted detection gaps, made self-documenting in threshold_basis instead of silent.

-- verify_grounding does not exist as a capability, so vg_verdict is NULL on every
-- hallucination_signal row by construction (hallucination_signal_liveness already WARNs on
-- this). Layers 2/3 (resp_vs_prompt_similarity, ungrounded_token_count) carry the signal today.
INSERT INTO mq_gmdf_dev.oil_obs.threshold_basis
  (check_name, threshold, basis, set_on, status)
SELECT * FROM VALUES
  ('hallucination_verify_grounding',
   'vg_verdict computed via verify_grounding (Layer 1)',
   'verify_grounding does not exist as a capability -- vg_verdict is NULL on every '
   'hallucination_signal row by construction. Layers 2/3 (similarity + ungrounded token '
   'count) carry the signal today. Re-promote once verify_grounding ships.',
   DATE '2026-09-01', 'not yet active')
  AS t(check_name, threshold, basis, set_on, status)
WHERE NOT EXISTS (
  SELECT 1 FROM mq_gmdf_dev.oil_obs.threshold_basis
  WHERE check_name = 'hallucination_verify_grounding');

-- rapid_human_correction's ai_publish CTE requires shift_type <> '' AND batch_nbr IS NOT NULL;
-- every dsa_* capability writes NULL/'' for shift_date/shift_type/batch_nbr in v_llm_bronze, so
-- it is structurally excluded, not broken. 0 rows is the correct state today.
INSERT INTO mq_gmdf_dev.oil_obs.threshold_basis
  (check_name, threshold, basis, set_on, status)
SELECT * FROM VALUES
  ('rapid_human_correction',
   'AI publish and ISH correction match on (shift_date, shift_type, batch_nbr) within 10 minutes',
   'dsa_* capabilities write NULL or an empty string for shift_date/shift_type/batch_nbr, so '
   'ai_publish excludes them entirely -- 0 rows is expected, not a failure. Revisit after '
   '2026-10-01; ptof_obs_verification.ipynb WARNs if still 0 rows past that date.',
   DATE '2026-09-01', 'not yet active')
  AS t(check_name, threshold, basis, set_on, status)
WHERE NOT EXISTS (
  SELECT 1 FROM mq_gmdf_dev.oil_obs.threshold_basis
  WHERE check_name = 'rapid_human_correction');

In [ ]:
%sql
-- SAFE to re-run: WHERE NOT EXISTS-guarded per capability -- unlike the INSERT OVERWRITE cell
-- above, re-running this cannot wipe the 6 hand-curated rows. Phase 4 item 3 of
-- agent_obs_implementation_plan.md: registry drift causing the confirmed 406-violation dev
-- backtest failure and the P1.2 "registry covers every capability" verification FAIL.
--
-- transport/model_config observed directly in v_llm_bronze (read-only query, 2026-09-01) --
-- recording reality, not a judgment call. is_gxp_relevant = false for all four, confirmed with
-- user -- not inferring GxP-relevance from dsa_session_summary's name alone. owner defaults to
-- 'unassigned', the existing convention, pending real owners.
INSERT INTO mq_gmdf_dev.oil_obs.capability_registry
  (capability, is_generative, is_gxp_relevant, expected_min_daily, required_fields, owner,
   active, notes, silence_grace_hours)
SELECT * FROM VALUES
  ('dsa_copilot_step', true, false, 0, array(), 'unassigned', true,
     'cortex · registry-drift fix, added 2026-09-01 · observed: test-bot-claude-v1 (605), '
     'vertex26 (59), cortex-claude46 (4), my-agent-triage-rag-ag-grp-dev (2)', NULL),
  ('dsa_session_summary', true, false, 0, array(), 'unassigned', true,
     'cortex · registry-drift fix, added 2026-09-01 · observed: test-bot-claude-v1 (17), '
     'vertex26 (6)', NULL),
  ('probe', true, false, 0, array(), 'unassigned', true,
     'cortex · registry-drift fix, added 2026-09-01 · observed: test-bot-claude-v1 (1)', NULL),
  ('saa_insight', true, false, 0, array(), 'unassigned', true,
     'cortex · registry-drift fix, added 2026-09-01 · observed: test-bot-claude-v1 (115), '
     'spe-claude-sonnet-46 (34), cortex-claude46 (6), my-agent-triage-rag-ag-grp-dev (1)', NULL)
  AS t(capability, is_generative, is_gxp_relevant, expected_min_daily, required_fields, owner,
       active, notes, silence_grace_hours)
WHERE NOT EXISTS (
  SELECT 1 FROM mq_gmdf_dev.oil_obs.capability_registry
  WHERE capability = 'dsa_copilot_step');

In [ ]:
%sql
-- SAFE to re-run: WHERE NOT EXISTS-guarded per capability, same idempotent pattern as the
-- capability_registry insert above. Completes Phase 4 item 3 of
-- agent_obs_implementation_plan.md: the '0.6b dev backtest = 0 violations' check in
-- ptof_obs_verification.ipynb joins v_llm_bronze to runtime_allowlist (not
-- capability_registry) by capability, so without a runtime_allowlist row these 4
-- capabilities would still read as unknown_capability violations even after the
-- capability_registry insert above.
--
-- transport/model_config/scheduler_run observed directly in v_llm_bronze (read-only
-- query, 2026-09-01) -- recording reality, not a judgment call. dev only: this data is
-- entirely from the mq_gmdf_dev catalog, so a prod row would be invented, not observed.
INSERT INTO mq_gmdf_dev.oil_obs.runtime_allowlist
  (environment, capability, allowed_transports, allowed_model_configs, allowed_scheduler_runs)
SELECT * FROM VALUES
  ('dev', 'dsa_copilot_step', array('cortex'),
     array('test-bot-claude-v1','vertex26','cortex-claude46','my-agent-triage-rag-ag-grp-dev'),
     array('live')),
  ('dev', 'dsa_session_summary', array('cortex'),
     array('test-bot-claude-v1','vertex26'), array('live')),
  ('dev', 'probe', array('cortex'), array('test-bot-claude-v1'), array('live')),
  ('dev', 'saa_insight', array('cortex'),
     array('test-bot-claude-v1','spe-claude-sonnet-46','cortex-claude46',
           'my-agent-triage-rag-ag-grp-dev'), array('live'))
  AS t(environment, capability, allowed_transports, allowed_model_configs,
       allowed_scheduler_runs)
WHERE NOT EXISTS (
  SELECT 1 FROM mq_gmdf_dev.oil_obs.runtime_allowlist
  WHERE environment = 'dev' AND capability = 'dsa_copilot_step');